# FFAI DAG Validation

This notebook demonstrates FFAI's dependency graph validation system, ported
from Plico's orchestrator. The DAG system provides:

1. **Cycle detection** -- catches circular dependencies before execution
2. **Dependency level assignment** -- topological ordering for parallel scheduling
3. **Edge source tracking** -- distinguishes `history` edges from implicit
   `condition` edges
4. **`validate_graph()`** -- high-level API to validate a prompt graph

These features let you verify the correctness of a multi-step workflow
**before** spending any API tokens.

In [ ]:
import os
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from dotenv import load_dotenv  # noqa: E402

load_dotenv()

from ffai.Clients.FFLiteLLMClient import FFLiteLLMClient  # noqa: E402
from ffai.core.execution_state import ExecutionState  # noqa: E402
from ffai.core.graph import (  # noqa: E402
    build_execution_graph_with_edges,
    get_ready_prompts,
)
from ffai.FFAI import FFAI  # noqa: E402

api_key = os.getenv('MISTRAL_API_KEY')
if not api_key:
    raise RuntimeError('Set MISTRAL_API_KEY in your environment or .env file')

client = FFLiteLLMClient(
    model_string='mistral/mistral-small-latest',
    api_key=api_key,
    temperature=0.3,
    max_tokens=256,
)

ffai = FFAI(client)

print('FFAI initialized with LiteLLM Mistral Small client')
print('Available: FFAI.validate_graph(), build_execution_graph_with_edges(), get_ready_prompts()')

<div class="page-break"></div>

---

## Example 1: Build a Simple DAG

A three-step workflow where `context` has no dependencies, `problem` depends on
`context`, and `solution` depends on both. The graph builder assigns levels
based on topological ordering:

- Level 0: no dependencies (can run immediately)
- Level N: max(dependency levels) + 1

In [ ]:
prompts = [
    {"sequence": 0, "prompt_name": "context", "prompt": "I run a coffee shop.", "history": []},
    {"sequence": 1, "prompt_name": "problem", "prompt": "My electricity bill is too high.", "history": ["context"]},
    {"sequence": 2, "prompt_name": "solution", "prompt": "Suggest 3 ways to reduce costs.", "history": ["context", "problem"]},
]

graph = build_execution_graph_with_edges(prompts)

print(f"Nodes: {len(graph.nodes)}")
print(f"Edges: {len(graph.edges)}")
print(f"Max level: {graph.max_level}")
print()

for seq, node in sorted(graph.nodes.items()):
    name = node.get_prompt_name() or f"seq_{seq}"
    deps = node.dependencies
    print(f"  {name:12s}  level={node.level}  dependencies={deps or 'none'}")

<div class="page-break"></div>

---

## Example 2: Inspect Dependency Edges

Each edge records its source (`history` or `condition`) and the sequence
numbers it connects. Condition edges also carry the original condition text.

In [ ]:
print("Dependency edges:")
print()

for edge in graph.edges:
    from_name = graph.nodes[edge.from_seq].get_prompt_name()
    to_name = graph.nodes[edge.to_seq].get_prompt_name()
    print(f"  {from_name:12s} -> {to_name:12s}  [{edge.source}]")

<div class="page-break"></div>

---

## Example 3: Condition-Sourced Implicit Edges

When a prompt's `condition` field references another prompt via
`{{name.property}}`, the graph builder creates an **implicit dependency edge**
even if the user didn't declare it in `history`. This catches undeclared
dependencies.

The edge `source` field distinguishes:
- `"history"` -- explicitly declared via the `history` parameter
- `"condition"` -- inferred from `{{name.property}}` in a condition expression

In [ ]:
prompts_with_conditions = [
    {"sequence": 0, "prompt_name": "fetch", "prompt": "Retrieve the data", "history": []},
    {
        "sequence": 1,
        "prompt_name": "process",
        "prompt": "Process the data",
        "history": [],
        "condition": '{{fetch.status}} == "success"',
    },
]

graph2 = build_execution_graph_with_edges(prompts_with_conditions)

print("Edges:")
for edge in graph2.edges:
    from_name = graph2.nodes[edge.from_seq].get_prompt_name()
    to_name = graph2.nodes[edge.to_seq].get_prompt_name()
    cond = f' condition="{edge.condition_text}"' if edge.condition_text else ""
    print(f"  {from_name} -> {to_name}  [{edge.source}]{cond}")

print()
print(f"process dependencies: {graph2.nodes[1].dependencies}")
print(f"process level: {graph2.nodes[1].level}")

<div class="page-break"></div>

---

## Example 4: Cycle Detection

The graph builder raises `ValueError` when a dependency cycle is detected.
This prevents infinite loops in execution. Three types of cycles are caught:

- Direct (A depends on B, B depends on A)
- Transitive (A -> C -> B -> A)
- Self-referencing (A depends on itself)

In [ ]:
circular_prompts = [
    {"sequence": 0, "prompt_name": "a", "prompt": "First", "history": ["b"]},
    {"sequence": 1, "prompt_name": "b", "prompt": "Second", "history": ["a"]},
]

try:
    build_execution_graph_with_edges(circular_prompts)
except ValueError as e:
    print(f"Cycle detected: {e}")

<div class="page-break"></div>

---

## Example 5: Parallel-Ready Execution with `get_ready_prompts`

The `get_ready_prompts()` function returns nodes whose dependencies are all
completed, sorted by level and sequence. Combined with `ExecutionState`,
this enables level-based parallel scheduling.

In this example, `a` and `b` are independent (both level 0), while `c`
depends on both. After `a` and `b` complete, `c` becomes ready.

In [ ]:
prompts_parallel = [
    {"sequence": 0, "prompt_name": "a", "prompt": "Task A", "history": []},
    {"sequence": 1, "prompt_name": "b", "prompt": "Task B", "history": []},
    {"sequence": 2, "prompt_name": "c", "prompt": "Synthesize", "history": ["a", "b"]},
]

graph3 = build_execution_graph_with_edges(prompts_parallel)
state = ExecutionState()

print("=== Round 1: Initial ===")
ready = get_ready_prompts(state, graph3.nodes)
print(f"Ready: {[n.get_prompt_name() for n in ready]}")

for node in ready:
    state.completed.add(node.sequence)
    state.in_progress.discard(node.sequence)

print()
print("=== Round 2: After a and b complete ===")
ready = get_ready_prompts(state, graph3.nodes)
print(f"Ready: {[n.get_prompt_name() for n in ready]}")

for node in ready:
    state.completed.add(node.sequence)

print()
print("=== Round 3: After c completes ===")
ready = get_ready_prompts(state, graph3.nodes)
print(f"Ready: {[n.get_prompt_name() for n in ready]}")
print(f"All completed: {len(state.completed) == len(graph3.nodes)}")

<div class="page-break"></div>

---

## Example 6: `validate_graph()` -- High-Level Validation API

FFAI's `validate_graph()` method wraps the graph builder and adds
**undeclared dependency warnings**. It detects when a condition references
a prompt that is NOT listed in `history`, flagging it as a potential bug.

This lets you catch issues before executing a workflow.

In [ ]:
workflow = [
    {"prompt_name": "fetch", "prompt": "Retrieve the data"},
    {
        "prompt_name": "process",
        "prompt": "Process the results",
        "condition": '{{fetch.status}} == "success"',
    },
    {
        "prompt_name": "fallback",
        "prompt": "Generate sample data",
        "condition": '{{fetch.status}} == "failed"',
    },
    {
        "prompt_name": "report",
        "prompt": "Write a summary",
        "history": ["fetch", "process"],
        "condition": '{{process.status}} == "success"',
    },
]

graph, warnings = ffai.workflow.validate_graph(workflow)

print(f"Validated {len(workflow)} prompts")
print(f"Max dependency level: {graph.max_level}")
print(f"Total edges: {len(graph.edges)}")
print()

print("Nodes:")
for seq, node in sorted(graph.nodes.items()):
    name = node.get_prompt_name() or f"seq_{seq}"
    deps = {graph.nodes[d].get_prompt_name() for d in node.dependencies}
    print(f"  {name:12s}  level={node.level}  depends_on={deps or '{}'}")

print()
print(f"Warnings ({len(warnings)}):")
for w in warnings:
    print(f"  - {w}")

<div class="page-break"></div>

---

## Example 7: Valid Graph (No Warnings)

When all condition references are also declared in `history`, no warnings
are produced. The `report` step below declares `process` in both `history`
and `condition`, so it passes cleanly.

In [ ]:
clean_workflow = [
    {"prompt_name": "fetch", "prompt": "Retrieve the data"},
    {
        "prompt_name": "process",
        "prompt": "Process the results",
        "history": ["fetch"],
        "condition": '{{fetch.status}} == "success"',
    },
]

graph, warnings = ffai.workflow.validate_graph(clean_workflow)

print(f"Nodes: {len(graph.nodes)}")
print(f"Max level: {graph.max_level}")
print(f"Warnings: {len(warnings)}")

for seq, node in sorted(graph.nodes.items()):
    name = node.get_prompt_name() or f"seq_{seq}"
    deps = {graph.nodes[d].get_prompt_name() for d in node.dependencies}
    print(f"  {name:12s}  level={node.level}  depends_on={deps or '{}'}")

<div class="page-break"></div>

---

## Example 8: Detecting a Cycle via `validate_graph`

`validate_graph()` raises `ValueError` immediately when a cycle is found,
preventing a broken workflow from being executed.

In [ ]:
circular_workflow = [
    {"prompt_name": "a", "prompt": "Step A", "history": ["b"]},
    {"prompt_name": "b", "prompt": "Step B", "history": ["a"]},
]

try:
    ffai.workflow.validate_graph(circular_workflow)
    print("No error raised (unexpected)")
except ValueError as e:
    print(f"Cycle detected: {e}")

<div class="page-break"></div>

---

## Summary

| API | Purpose |
|-----|---------|
| `ffai.workflow.validate_graph(prompts)` | Validate a prompt graph; returns `(ExecutionGraph, warnings)` |
| `build_execution_graph_with_edges(prompts)` | Low-level graph builder with edge metadata |
| `get_ready_prompts(state, nodes)` | Find nodes ready for execution |
| `ExecutionState` | Thread-safe state tracking for parallel execution |

| Edge source | Meaning |
|-------------|---------|
| `"history"` | Explicitly declared via `history=["name"]` |
| `"condition"` | Inferred from `{{name.property}}` in a condition expression |

| Validation result | Meaning |
|-------------------|--------|
| `ExecutionGraph` | Nodes with levels + edges with sources |
| `warnings` list | Undeclared condition dependencies |
| `ValueError` | Dependency cycle detected |